# Deep-learning models: RealMLP blended into the best model

`neural_net_model.ipynb` showed that a plain neural network adds very little to our best model. This notebook tries two **modern deep-learning models for tables** from the `pytabkit` library (**RealMLP**, a neural network with tuned defaults, and **TabM**, an efficient ensemble of 32 small networks), keeps the one that helps, and blends it into `candidate_nn_w05.csv`:

```
prediction = (1 - w) x candidate_nn_w05.csv + w x RealMLP (average of several seeds)      [+ shift on rows with previous_usage blank]
             w = REALMLP_WEIGHT_COMPLETE_ROWS on rows with no blank input, REALMLP_WEIGHT_BLANK_ROWS on the others
(candidate_nn_w05.csv = 95% candidate_v8.csv + 5% scikit-learn neural network, from neural_net_model.ipynb)
```

| Settings | File | Public RMSE |
|---|---|---|
| 3 seeds, w = 0.07 everywhere | `candidate_dl.csv` | **3.26340** |
| 10 seeds, w = 0.20 on complete rows, 0 on blank rows | `candidate_gated.csv` | 3.26880 |
| 10 seeds, w = 0.13 everywhere | `candidate_dl13.csv` | 3.25670 |
| 3 seeds, w = 0.07 everywhere, shift +0.8 | `candidate_prevfix.csv` | 3.25850 |
| 10 seeds, w = 0.13 everywhere, shift +0.8 (**default**) | `candidate_dl13_prevfix.csv` | _not submitted yet_ (predicted ≈ 3.2512) |

**Environment.** This notebook needs PyTorch and `pytabkit`, which are **not** in the platform environment. Run it with the `datathon-dl-env` interpreter (`/Users/ethanis/Downloads/datathon-dl-env/bin/python`; in VS Code: *Select Kernel → Python Environments*). It reads `candidate_nn_w05.csv`, which `neural_net_model.ipynb` writes.

## Part A: settings, data and features

In [1]:
import logging
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from pytabkit import RealMLP_TD_Regressor
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')
for logger_name in ['lightning', 'lightning.pytorch', 'lightning.fabric', 'pytorch_lightning']:
    logging.getLogger(logger_name).setLevel(logging.ERROR)      # hide PyTorch Lightning's progress messages

# ---------------- settings you might want to change ----------------
DATA_FOLDER = Path('.')
BASE_FILE = DATA_FOLDER / 'candidate_nn_w05.csv'    # what RealMLP is blended into
OUTPUT_FILE = DATA_FOLDER / 'candidate_dl13_prevfix.csv'
REALMLP_WEIGHT_COMPLETE_ROWS = 0.13   # RealMLP weight on test rows with no blank input
REALMLP_WEIGHT_BLANK_ROWS = 0.13      # ... and on rows with at least one blank (0.13 / 0.13 = what the public scores prefer)
PREVIOUS_USAGE_BLANK_SHIFT = 0.8      # added to rows whose previous_usage is blank (see Part B and Results; 0.0 -> candidate_dl13.csv)
REALMLP_SEEDS = list(range(10))       # final RealMLP = average of these seeds (about 30 seconds each)
REUSE_SAVED_SEEDS = True              # True: load experiments/realmlp_test_seed<N>.npy when it exists
                                      # (made by this same code; a retrain matches it to 1e-12)
RUN_CROSS_VALIDATION = False          # True: re-run RealMLP's 5-fold cross-validation for CV_SEEDS (about 3 minutes per seed)
CV_SEEDS = [0, 1, 2, 3, 4]            # False: score the saved fold predictions in experiments/
CPU_THREADS = 8

print('torch', torch.__version__, '| pandas', pd.__version__)

/Users/ethanis/Downloads/datathon-dl-env/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


torch 2.14.1 | pandas 3.0.6


In [2]:
train = pd.read_csv(DATA_FOLDER / 'train.csv')
test = pd.read_csv(DATA_FOLDER / 'test.csv')
test_inputs = test.drop(columns=['energy_usage'], errors='ignore')

NUMERIC_COLUMNS = ['temperature', 'humidity', 'occupancy', 'previous_usage']
CATEGORICAL_COLUMNS = ['building_id', 'building_type', 'day_of_week']
test_blank_patterns = test_inputs[NUMERIC_COLUMNS].isna().to_numpy()
print(f'train: {len(train):,} rows | test: {len(test):,} rows')


def with_test_like_blanks(rows, seed):
    """Copy of rows with extra blanks copied from random test rows (same as neural_net_model.ipynb)."""
    random_generator = np.random.default_rng(seed)
    blanks = test_blank_patterns[random_generator.integers(0, len(test_inputs), len(rows))]
    masked = rows.copy()
    masked[NUMERIC_COLUMNS] = masked[NUMERIC_COLUMNS].mask(blanks)
    return masked


class RealMLPFeatures:
    """Raw inputs for RealMLP. Blanks get the building x hour median of the training rows, plus a 'was blank' flag."""

    def fit(self, rows):
        self.by_hour = {c: rows.groupby(['building_id', 'hour'])[c].median() for c in NUMERIC_COLUMNS}
        self.by_building = {c: rows.groupby('building_id')[c].median() for c in NUMERIC_COLUMNS}
        return self

    def transform(self, rows):
        features = pd.DataFrame(index=rows.index)
        for column in CATEGORICAL_COLUMNS:              # RealMLP learns an embedding for each category
            features[column] = rows[column].astype(str)
        features['hour'] = rows['hour'].astype(float)
        features['month'] = rows['month'].astype(float)
        features['weekend'] = rows['day_of_week'].isin(['Saturday', 'Sunday']).astype(float)
        features['hour_sin'] = np.sin(2 * np.pi * rows['hour'] / 24)
        features['hour_cos'] = np.cos(2 * np.pi * rows['hour'] / 24)
        features['month_sin'] = np.sin(2 * np.pi * rows['month'] / 12)
        features['month_cos'] = np.cos(2 * np.pi * rows['month'] / 12)
        lookup = pd.MultiIndex.from_arrays([rows['building_id'], rows['hour']])
        for column in NUMERIC_COLUMNS:
            fill = pd.Series(self.by_hour[column].reindex(lookup).to_numpy(), index=rows.index)
            fill = fill.fillna(rows['building_id'].map(self.by_building[column]))
            features[column] = rows[column].fillna(fill).astype(float)
            features[f'{column}_blank'] = rows[column].isna().astype(float)
        return features


def fit_predict_realmlp(training_rows, rows_to_predict, seed):
    """Fit one RealMLP (default 'TD' settings) and predict each DataFrame in rows_to_predict."""
    features = RealMLPFeatures().fit(training_rows)
    model = RealMLP_TD_Regressor(device='cpu', random_state=seed, n_threads=CPU_THREADS, verbosity=0)
    model.fit(features.transform(training_rows), training_rows['energy_usage'].to_numpy(), cat_col_names=CATEGORICAL_COLUMNS)
    return [model.predict(features.transform(rows)) for rows in rows_to_predict]

train: 8,000 rows | test: 3,000 rows


## Part B: cross-validation

Same five folds and the same test-like blanks as `neural_net_model.ipynb` (the blank masks were checked to be identical in both environments), so the fold predictions of every model can be compared and blended row by row. The best model's fold predictions come from `experiments/neural_net_cv.npz`.

Scores (as in `neural_net_model.ipynb`): **as is** = plain RMSE; **test-like blanks** = held-out rows get extra blanks copied from test rows; **weighted** = test-like blanks with rows weighted to the test set's building × event mix.

**Nested blend weights:** the weight is learned on four folds (weighted score) and applied to the fifth, so the gain is not flattered by tuning on the scored rows. **Gated** means RealMLP's weight depends on whether the row has any blank input (known for test rows too).

In [3]:
folds = list(KFold(5, shuffle=True, random_state=0).split(train))
truth = train['energy_usage'].to_numpy()
KINDS = ['as_is', 'test_blanks']


def cv_file(seed):
    return DATA_FOLDER / 'experiments' / ('dl_cv_realmlp.npz' if seed == 0 else f'dl_cv_realmlp_seed{seed}.npz')


if RUN_CROSS_VALIDATION:
    for seed in CV_SEEDS:
        realmlp_cv = {kind: np.zeros(len(train)) for kind in KINDS}
        for fold, (fit_positions, check_positions) in enumerate(folds):
            started = time.time()
            check_rows = train.iloc[check_positions].drop(columns='energy_usage')
            as_is, masked = fit_predict_realmlp(train.iloc[fit_positions], [check_rows, with_test_like_blanks(check_rows, seed=fold)], seed=seed)
            realmlp_cv['as_is'][check_positions], realmlp_cv['test_blanks'][check_positions] = as_is, masked
            print(f'seed {seed} fold {fold} done ({time.time() - started:.0f}s)')
        np.savez(cv_file(seed), **realmlp_cv)

saved = np.load(DATA_FOLDER / 'experiments' / 'neural_net_cv.npz')
assert np.array_equal(saved['truth'], truth)
row_weights = saved['row_weights']
best_cv = {k: 0.25 * saved[f'v4_{k}'] + 0.75 * saved[f'lightgbm_{k}'] for k in KINDS}
base_cv = {k: 0.95 * best_cv[k] + 0.05 * saved[f'neural_net_{k}'] for k in KINDS}       # candidate_nn_w05 recipe
realmlp_seed_cv = [np.load(cv_file(seed)) for seed in CV_SEEDS if cv_file(seed).exists()]
realmlp_avg_cv = {k: np.mean([s[k] for s in realmlp_seed_cv], axis=0) for k in KINDS}

# which held-out rows have a blank input, as is and after test-like blanks (needed for gating)
has_blank = {'as_is': train[NUMERIC_COLUMNS].isna().any(axis=1).to_numpy(), 'test_blanks': np.zeros(len(train), bool)}
for fold, (fit_positions, check_positions) in enumerate(folds):
    masked = with_test_like_blanks(train.iloc[check_positions], seed=fold)
    has_blank['test_blanks'][check_positions] = masked[NUMERIC_COLUMNS].isna().any(axis=1).to_numpy()


def three_scores(prediction):
    return {'as is': np.sqrt(np.mean((prediction['as_is'] - truth) ** 2)),
            'test-like blanks': np.sqrt(np.mean((prediction['test_blanks'] - truth) ** 2)),
            'weighted': np.sqrt(np.average((prediction['test_blanks'] - truth) ** 2, weights=row_weights))}


standalone = {'best model (candidate_v8 recipe)': best_cv, 'candidate_nn_w05 recipe': base_cv,
              'RealMLP, 1 seed': {k: realmlp_seed_cv[0][k] for k in KINDS},
              f'RealMLP, average of {len(realmlp_seed_cv)} seeds': realmlp_avg_cv}
for name, file_name in [('TabM', 'dl_cv_tabm.npz'), ('RealMLP + test-like blank copies', 'dl_cv_realmlp_aug.npz')]:
    if (DATA_FOLDER / 'experiments' / file_name).exists():
        loaded = np.load(DATA_FOLDER / 'experiments' / file_name)
        standalone[name] = {k: loaded[k] for k in KINDS}
print('Cross-validation RMSE of each model on its own (lower is better)')
display(pd.DataFrame({name: three_scores(p) for name, p in standalone.items()}).T.round(4))

rows_with_blanks = has_blank['test_blanks']
by_blanks = pd.DataFrame({name: {'rows with no blank': np.sqrt(np.mean((p['test_blanks'][~rows_with_blanks] - truth[~rows_with_blanks]) ** 2)),
                                 'rows with a blank': np.sqrt(np.mean((p['test_blanks'][rows_with_blanks] - truth[rows_with_blanks]) ** 2))}
                          for name, p in [('best model', best_cv), (f'RealMLP ({len(realmlp_seed_cv)} seeds)', realmlp_avg_cv)]}).T
print('RMSE with test-like blanks, split by whether the row has a blank')
display(by_blanks.round(3))

Cross-validation RMSE of each model on its own (lower is better)


,as is,test-like blanks,weighted
best model (candidate_v8 recipe),3.0742,3.2791,3.6394
candidate_nn_w05 recipe,3.0741,3.2789,3.6373
"RealMLP, 1 seed",3.2283,3.6016,4.0694
"RealMLP, average of 5 seeds",3.1813,3.5560,4.0258
TabM,3.2106,3.5661,4.0198
RealMLP + test-like blank copies,3.7997,4.0301,4.6230


RMSE with test-like blanks, split by whether the row has a blank


,rows with no blank,rows with a blank
best model,3.014,4.036
RealMLP (5 seeds),3.067,4.837


In [4]:
def blend(weight_complete, weight_blank):
    return {k: base_cv[k] + np.where(has_blank[k], weight_blank, weight_complete) * (realmlp_avg_cv[k] - base_cv[k]) for k in KINDS}


def nested_weights(gated):
    # weights learned on 4 folds (weighted score), applied to the 5th
    blended = {k: base_cv[k].copy() for k in KINDS}
    learned = []
    for fit_positions, check_positions in folds:
        weights = []
        for group in ([~has_blank['test_blanks'], has_blank['test_blanks']] if gated else [np.ones(len(train), bool)]):
            m = group[fit_positions]
            gap = (realmlp_avg_cv['test_blanks'] - base_cv['test_blanks'])[fit_positions][m]
            error = (truth - base_cv['test_blanks'])[fit_positions][m]
            w = row_weights[fit_positions][m]
            weights.append(float(np.clip(np.sum(w * gap * error) / np.sum(w * gap ** 2), 0, 1)))
        weight_complete, weight_blank = (weights * 2)[:2]
        learned.append((weight_complete, weight_blank))
        for k in KINDS:
            weight = np.where(has_blank[k][check_positions], weight_blank, weight_complete)
            blended[k][check_positions] = base_cv[k][check_positions] + weight * (realmlp_avg_cv[k] - base_cv[k])[check_positions]
    return blended, np.mean(learned, axis=0)


start_scores = three_scores(base_cv)
table = {}
ungated, (w_all, _) = nested_weights(gated=False)
table[f'one weight for all rows (nested: {w_all:.2f})'] = three_scores(ungated)
gated, (w_complete, w_blank) = nested_weights(gated=True)
table[f'gated (nested: {w_complete:.2f} complete / {w_blank:.2f} blank)'] = three_scores(gated)
for weight in (0.10, 0.15, 0.20, 0.25, 0.30):
    table[f'gated, fixed: {weight:.2f} on complete rows, 0 on blank rows'] = three_scores(blend(weight, 0.0))
gains = pd.DataFrame({name: {f'gain: {s}': start_scores[s] - v for s, v in scores.items()} for name, scores in table.items()}).T
print('Blending RealMLP into the candidate_nn_w05 recipe (positive gain = better)')
display(gains.round(4))

Blending RealMLP into the candidate_nn_w05 recipe (positive gain = better)


,gain: as is,gain: test-like blanks,gain: weighted
one weight for all rows (nested: 0.07),0.0015,0.0013,0.0018
gated (nested: 0.31 complete / 0.01 blank),0.0025,0.0017,0.0081
"gated, fixed: 0.10 on complete rows, 0 on blank rows",0.0030,0.0023,0.0064
"gated, fixed: 0.15 on complete rows, 0 on blank rows",0.0039,0.0030,0.0087
"gated, fixed: 0.20 on complete rows, 0 on blank rows",0.0043,0.0033,0.0103
"gated, fixed: 0.25 on complete rows, 0 on blank rows",0.0042,0.0033,0.0113
"gated, fixed: 0.30 on complete rows, 0 on blank rows",0.0038,0.0029,0.0117


### Real blanks are not random

The cross-validation above makes blanks by copying test blank patterns onto random training rows. That assumes blanks are random. The training set's **own** blanks show they are not, at least for `previous_usage`: when it is genuinely blank, every imputation-based model under-predicts energy by about 1.1, in every fold. Random blanks show no such bias. RealMLP sees an explicit "was blank" flag and learned most of this effect.

In [5]:
best_as_is = 0.25 * saved['v4_as_is'] + 0.75 * saved['lightgbm_as_is']
rows = {}
for column in NUMERIC_COLUMNS:
    real_blank = train[column].isna().to_numpy()
    random_blank = np.zeros(len(train), bool)
    for fold, (fit_positions, check_positions) in enumerate(folds):
        masked = with_test_like_blanks(train.iloc[check_positions], seed=fold)
        random_blank[check_positions] = masked[column].isna().to_numpy() & ~real_blank[check_positions]
    error_real, error_random = (best_as_is - truth)[real_blank], (best_cv['test_blanks'] - truth)[random_blank]
    rows[column] = {'real blanks: rows': real_blank.sum(),
                    'real blanks: best-model bias': error_real.mean(),
                    'real blanks: standard error': error_real.std() / np.sqrt(real_blank.sum()),
                    'RealMLP bias (real blanks)': (realmlp_avg_cv['as_is'] - truth)[real_blank].mean(),
                    'random blanks: best-model bias': error_random.mean(),
                    'random blanks: standard error': error_random.std() / np.sqrt(random_blank.sum())}
print('Average error (prediction - truth) on rows where the column is blank; negative = under-prediction')
display(pd.DataFrame(rows).T.round(2))

Average error (prediction - truth) on rows where the column is blank; negative = under-prediction


,real blanks: rows,real blanks: best-model bias,real blanks: standard error,RealMLP bias (real blanks),random blanks: best-model bias,random blanks: standard error
temperature,119.0,0.34,0.29,0.11,0.00,0.18
humidity,119.0,-0.26,0.26,-0.21,-0.11,0.17
occupancy,122.0,-0.27,0.42,-0.22,0.14,0.19
previous_usage,114.0,-1.12,0.36,-0.18,-0.04,0.20


## Part C: final fit and submission file

RealMLP is fitted on all of `train.csv` once per seed in `REALMLP_SEEDS` (or the saved test predictions of the same fits are loaded), the seeds are averaged, and the result is blended into `candidate_nn_w05.csv`: weight `REALMLP_WEIGHT_COMPLETE_ROWS` on test rows with no blank input, `REALMLP_WEIGHT_BLANK_ROWS` on the others. Rows whose `previous_usage` is blank then get `PREVIOUS_USAGE_BLANK_SHIFT` added.

In [6]:
base = pd.read_csv(BASE_FILE)
assert base['id'].equals(test['id']), f'{BASE_FILE.name} rows are not in test.csv order'

seed_predictions = []
for seed in REALMLP_SEEDS:
    saved_file = DATA_FOLDER / 'experiments' / f'realmlp_test_seed{seed}.npy'
    if REUSE_SAVED_SEEDS and saved_file.exists():
        seed_predictions.append(np.load(saved_file))
        continue
    started = time.time()
    seed_predictions.append(fit_predict_realmlp(train, [test_inputs], seed=seed)[0])
    np.save(saved_file, seed_predictions[-1])
    print(f'RealMLP seed {seed} trained ({time.time() - started:.0f}s)')
realmlp_prediction = np.mean(seed_predictions, axis=0)
print(f'RealMLP: average of {len(seed_predictions)} seeds | typical difference between one seed and the average: '
      f'{np.sqrt(np.mean((np.array(seed_predictions) - realmlp_prediction) ** 2)):.3f}')

RealMLP: average of 10 seeds | typical difference between one seed and the average: 0.832


In [7]:
complete_rows = test_inputs[NUMERIC_COLUMNS].notna().all(axis=1).to_numpy()
realmlp_weight = np.where(complete_rows, REALMLP_WEIGHT_COMPLETE_ROWS, REALMLP_WEIGHT_BLANK_ROWS)
final_prediction = (1 - realmlp_weight) * base['prediction'].to_numpy() + realmlp_weight * realmlp_prediction
final_prediction = final_prediction + PREVIOUS_USAGE_BLANK_SHIFT * test_inputs['previous_usage'].isna().to_numpy()
assert len(final_prediction) == len(test) and np.isfinite(final_prediction).all()
submission = pd.DataFrame({'id': test['id'], 'prediction': final_prediction})
submission.to_csv(OUTPUT_FILE, index=False)

print(f'wrote {OUTPUT_FILE.name}: {len(submission):,} rows | blanks: {submission["prediction"].isna().sum()} | '
      f'RealMLP weight {REALMLP_WEIGHT_COMPLETE_ROWS:.2f} on {complete_rows.sum():,} complete rows, '
      f'{REALMLP_WEIGHT_BLANK_ROWS:.2f} on {(~complete_rows).sum():,} rows with blanks | '
      f'shift {PREVIOUS_USAGE_BLANK_SHIFT:+.2f} on {test_inputs["previous_usage"].isna().sum()} rows with previous_usage blank')
for name in ['candidate_v8.csv', 'candidate_dl13.csv', 'candidate_prevfix.csv']:
    if (DATA_FOLDER / name).exists():
        other = pd.read_csv(DATA_FOLDER / name)['prediction'].to_numpy()
        print(f'typical change from {name}: {np.sqrt(np.mean((final_prediction - other) ** 2)):.3f}')
submission.head()

wrote candidate_dl13_prevfix.csv: 3,000 rows | blanks: 0 | RealMLP weight 0.13 on 2,453 complete rows, 0.13 on 547 rows with blanks | shift +0.80 on 203 rows with previous_usage blank
typical change from candidate_v8.csv: 0.301
typical change from candidate_dl13.csv: 0.208
typical change from candidate_prevfix.csv: 0.110


,id,prediction
0,EC010541,71.781721
1,EC009755,66.146059
2,EC008886,56.144780
3,EC009317,64.561719
4,EC008958,21.277637


## Results

**Public leaderboard so far**

| File | Recipe | Public RMSE |
|---|---|---|
| `candidate_v8.csv` | best model | 3.27026 |
| `candidate_nn.csv` | + 15% scikit-learn network | 3.27148 |
| `candidate_dl.csv` | + 5% scikit-learn network + 7% RealMLP (3 seeds), all rows | **3.26340** |
| `candidate_gated.csv` | + 5% network + 20% RealMLP (10 seeds), complete rows only | 3.26880 |
| `candidate_dl13.csv` | + 5% network + 13% RealMLP (10 seeds), all rows | 3.25670 |
| `candidate_prevfix.csv` | `candidate_dl.csv` + 0.8 on rows with `previous_usage` blank | 3.25850 |
| **`candidate_dl13_prevfix.csv`** | **`candidate_dl13.csv` + 0.8 on rows with `previous_usage` blank** | _not submitted_ (predicted ≈ 3.2512) |

**Cross-validation and the leaderboard disagree about rows with blanks.**

* Cross-validation says RealMLP hurts on rows with blanks (4.84 vs 4.04 RMSE) and should be used only on complete rows (gated). The gated file scored **worse** publicly (3.26880 vs 3.26340).
* Solving the four public scores for RealMLP's effect by row type (same method as in `neural_net_model.ipynb`) gives a best public weight of about **0.13 on complete rows and 0.13 on blank rows**. On real test rows with blanks, RealMLP's differences from the best model line up with the best model's errors, which the simulated blanks never showed.
* **Likely reason:** cross-validation makes blanks by copying test blank patterns onto random rows, which assumes blanks are random. The real test blanks don't look like that: 18% of test rows have a blank (6% in train), and 7.7% have two or more (0.1% in train).

**Real blanks carry information** (table above). When `previous_usage` is genuinely blank in `train.csv`, every imputation-based model under-predicts energy by **1.12** (standard error 0.36; negative in all five folds), while random blanks show no bias (−0.04). The imputers fill in a typical value, but real blanks apparently hide higher-than-typical usage. RealMLP, which sees a "was blank" flag, is nearly unbiased there (−0.18). The other columns show no significant effect.

`candidate_prevfix.csv` tested whether test blanks behave the same way, by adding 0.8 to the 203 test rows with `previous_usage` blank. **They do:** its public score (3.25850 vs 3.26340) means `candidate_dl.csv` under-predicts those public rows by **0.70** on average (0.67–0.73 depending on how many of them are in the public part), close to the 1.12 ± 0.36 measured in `train.csv`. For `candidate_dl13.csv` the public estimate is 0.73. Shifts of 0.7 and 0.8 are within 0.0001 of each other on the public rows, and 0.8 sits between the public and training estimates, so the default is **0.8**.

**RealMLP weight:** with the `candidate_dl13.csv` score, the public-best RealMLP weight is about 0.17, but it would gain only about 0.0009 over 0.13, while cross-validation preferred less (0.07–0.09). The default stays at **0.13** rather than fitting the public rows further.

**Also tried, did not help**

* **TabM**: no gain alone or added to RealMLP.
* **Training RealMLP on test-like blank copies**: worse, because pytabkit's internal validation split leaks copies of training rows.
* **Different RealMLP weights for unusual "event" rows**: worse in nested cross-validation.
* **Deep-learning imputation**: a masked autoencoder (a neural network that learns to reconstruct hidden cells from the rest of the row) fills occupancy much more accurately than the LightGBM imputers (RMSE 23 vs 34). But swapped into the LightGBM-imputed model it did not improve energy predictions (−0.007 to +0.006 depending on the score). Details: `experiments/c_with_dae_cv.npz`.
* **Linking rows in time** (a row's `previous_usage` = another row's energy one hour earlier): matches occur at chance level only (0.3%), and the ids are shuffled.